[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_ConformalARIMA/VaR_CEE_ConformalARIMA.ipynb)

# VaR_CEE_ConformalARIMA

Implements ARIMA(1,0,0) with conformal prediction for VaR and ES forecasting.
Uses rolling 250-day AR(1) model with conformal quantiles of in-sample residuals
to construct prediction intervals.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
!pip install -q statsmodels

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
ROLLING_WINDOW = 250
STRIDE = 1  # R1: full daily run
SEED = 42

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

W = ROLLING_WINDOW
np.random.seed(SEED)

OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

## Upload Data

Upload the 10 raw CSV files for index and FX series.

In [ ]:
from google.colab import files
print("Upload the 10 raw CSV files (BET.csv, WIG20.csv, PX.csv, BUX.csv, SOFIX.csv,")
print("EURRON.csv, EURPLN.csv, EURCZK.csv, EURHUF.csv, USDBGN.csv):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files")

# Compute log returns
returns_dict = {}
for country, raw_id, series_id, stype in get_all_series():
    csv_file = f"{raw_id}.csv"
    if csv_file not in uploaded:
        print(f"  WARNING: {csv_file} not found, skipping {series_id}")
        continue
    df = pd.read_csv(csv_file, parse_dates=["Date"], index_col="Date").sort_index()
    log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
    log_ret.name = series_id
    returns_dict[series_id] = log_ret
    print(f"  {series_id}: {len(log_ret)} obs")
print(f"\nTotal: {len(returns_dict)} series")

In [ ]:
def run_arima_conformal(returns, oos_dates):
    """ARIMA + Conformal Prediction: rolling 250-day AR(1) with conformal residual quantiles."""
    results = []
    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < W:
            continue
        train = returns.iloc[loc - W:loc]
        y_true = returns.iloc[loc]

        try:
            from statsmodels.tsa.ar_model import AutoReg
            model = AutoReg(train.values, lags=1).fit()
            fitted = model.fittedvalues
            resids = train.values[1:] - fitted
            pred_next = model.predict(start=len(train), end=len(train))[0]
        except Exception:
            pred_next = train.mean()
            resids = (train - train.rolling(2).mean()).dropna().values

        row = {"date": date, "y_true": y_true}
        for alpha in VAR_ALPHAS:
            row[f"var_{alpha}"] = pred_next + np.quantile(resids, alpha)
        q_es = np.quantile(resids, ES_ALPHA)
        tail_r = resids[resids <= q_es]
        row["es_2p5pct"] = pred_next + (np.mean(tail_r) if len(tail_r) > 0 else q_es)
        results.append(row)

        if (i + 1) % 200 == 0:
            print(f"    ARIMA [{i+1}/{len(oos_dates)}]")

    return pd.DataFrame(results)

In [ ]:
print("=" * 60)
print("ARIMA-Conformal (rolling 250-day AR(1) + conformal)")
print("=" * 60)

all_results = {}
for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        continue
    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]
    print(f"\n[{series_id}] {len(returns)} obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_arima_conformal(returns, oos_dates)
    col_map = {f"var_{a}": f"var_{str(a).replace('0.', '')}pct" for a in VAR_ALPHAS}
    df_result = df_result.rename(columns=col_map)
    out_path = OUT_DIR / f"ARIMA-Conformal_{series_id}.csv"
    df_result.to_csv(out_path, index=False, float_format="%.8f")
    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {time.time()-t0:.1f}s")

print("\n=== ARIMA-Conformal complete ===")

In [ ]:
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue
    df_plot = all_results[series_id]
    var_col = "var_01pct"
    if var_col not in df_plot.columns:
        continue
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4, color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6, color="green", linestyle="--", label="ARIMA+CP VaR 1%")
    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"], df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")
    n_b = breaches.sum()
    ax.set_title(f"ARIMA-Conformal VaR 1% \u2014 {series_id} (breaches: {n_b}/{len(df_plot)} = {100*n_b/len(df_plot):.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from google.colab import files
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("ARIMA-Conformal_*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("ARIMA_Conformal_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("ARIMA_Conformal_results.zip")